# Task 3 — A/B Hypothesis Testing

Null hypotheses (alpha = 0.05):
1. H0: No risk difference across provinces.
2. H0: No risk difference between zip codes.
3. H0: No margin difference between zip codes.
4. H0: No risk difference between Women and Men.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd

from src.data_loader import load_insurance_data, add_derived_metrics
from src.hypothesis_tests import (
    chi_squared_frequency, t_test_numeric, z_test_proportions,
    anova_across_groups, results_table,
)

df = add_derived_metrics(load_insurance_data('../data/insurance_data.csv'))
df.shape

## H1: Risk across provinces
KPI: claim frequency + claim severity. Use ANOVA across all provinces, then pairwise.

In [ ]:
results = []

results.append(anova_across_groups(
    df, 'Province', 'TotalClaims',
    hypothesis='H1: No risk diff across provinces (severity)',
    only_claims=True,
))

# Pairwise example: pick the two largest provinces
top_provinces = df['Province'].value_counts().head(2).index.tolist()
results.append(chi_squared_frequency(
    df, 'Province', top_provinces[0], top_provinces[1],
    hypothesis=f'H1: {top_provinces[0]} vs {top_provinces[1]} (frequency)',
))

## H2 & H3: Zip-code risk and margin
Restrict to zip codes with sufficient exposure (e.g., >= 200 policies) and pick two balanced ones.

In [ ]:
zip_counts = df['PostalCode'].value_counts()
candidate_zips = zip_counts[zip_counts >= 200].index.tolist()[:2]

if len(candidate_zips) == 2:
    a, b = candidate_zips
    results.append(chi_squared_frequency(
        df, 'PostalCode', a, b,
        hypothesis=f'H2: zip {a} vs {b} (frequency)',
    ))
    results.append(t_test_numeric(
        df, 'PostalCode', a, b, 'Margin',
        hypothesis=f'H3: zip {a} vs {b} (margin)',
    ))

## H4: Gender risk
Two-proportion z-test for claim frequency + Welch t-test for severity (claims-only).

In [ ]:
results.append(z_test_proportions(
    df, 'Gender', 'Male', 'Female',
    hypothesis='H4: Men vs Women (frequency)',
))
results.append(t_test_numeric(
    df, 'Gender', 'Male', 'Female', 'TotalClaims',
    hypothesis='H4: Men vs Women (severity)',
    only_claims=True,
))

## Results

In [ ]:
results_table(results)

## Business interpretation
For each **rejected** H0, document:
- the observed direction (which group is riskier / more profitable),
- the magnitude (effect size + business units),
- the pricing/marketing implication for ACIS.